# KWISMO — Collecte de donnees (scraping + OCR)

Ce notebook n'ecrit aucune logique lui-meme : il appelle uniquement le code de `src/data/` (scrape.py, scrape_social.py, ocr.py, metrics.py). Fonctionne en local (venv deja actif, on est deja dans kwismo-ai/) comme sur Colab (clone + installe automatiquement).

In [ ]:
import sys

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

print("Environnement :", "Colab" if ON_COLAB else "local")

In [ ]:
if ON_COLAB:
    !git clone https://github.com/newtonachonduh46/kwismo.git
    %cd kwismo/kwismo-ai
    !grep -v '^torch' requirements.txt > /tmp/requirements-colab.txt
    !pip install -q -r /tmp/requirements-colab.txt
    !playwright install chromium
else:
    print("Local : venv deja actif, dependances deja installees (pip install -e .).")

## Configuration

`.env` n'est jamais versionne. En local il existe deja. Sur Colab, recree-le (comptes de scraping optionnels — sans eux, `scrape_social.run()` ne fait simplement rien).

In [ ]:
if ON_COLAB:
    %%writefile .env
MODEL_DIR="./models"
HF_MODEL_NAME="Davlan/afro-xlmr-base"
# FACEBOOK_ACCOUNTS="user1:pass1,user2:pass2"  # decommenter si besoin (voir Colab Secrets)

In [ ]:
import asyncio

from src.data import scrape, scrape_social, ocr, metrics

## 1. Sources texte (decouverte dynamique via recherche web)

In [ ]:
resume_texte = await scrape.run()
resume_texte

## 2. Facebook / Instagram / X (si des comptes sont configures dans .env)

In [ ]:
resume_social = await scrape_social.run()
resume_social

## 3. OCR des captures collectees

In [ ]:
nb_images_ocr = ocr.process_pending_images()
print(f"{nb_images_ocr} image(s) OCRisee(s).")

## 4. Metriques — evolution de la collecte

In [ ]:
plots_dir = metrics.generate_report()
historique = metrics.load_history()
print(f"{len(historique)} execution(s) enregistree(s) au total.")
print(f"Graphes sauvegardes dans {plots_dir}")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(plots_dir / "evolution_collecte.png")))
display(Image(filename=str(plots_dir / "erreurs_par_run.png")))

## 5. Publier les donnees collectees (Git)

`data/raw/scraped/messages.jsonl` et `data/interim/metrics/` sont versionnes — un `git push` les rend disponibles a toute l'equipe (voir `COLAB.md`).

In [ ]:
if ON_COLAB:
    !git config user.email "toi@exemple.com"
    !git config user.name "Ton Nom"
    !git add data/raw/scraped/*.jsonl data/interim/metrics/ data/interim/known_domains.json
    !git commit -m "Collecte de donnees (scraping) - session Colab"
    # !git push https://<TOKEN>@github.com/newtonachonduh46/kwismo.git main